# Flask – Practice Exercises

Hands-on exercises for Flask, following `g_fw-flask.ipynb`.

Use `app.test_client()` to send requests without starting a server.

### Contents

- [Exercise 1 – Hello World Route](#exercise-1--hello-world-route)
- [Exercise 2 – URL Path and Query Parameters](#exercise-2--url-path-and-query-parameters)
- [Exercise 3 – JSON Endpoints and the Request Object](#exercise-3--json-endpoints-and-the-request-object)
- [Exercise 4 – Data Validation with Pydantic](#exercise-4--data-validation-with-pydantic)
- [Exercise 5 – In-Memory Trading Endpoints](#exercise-5--in-memory-trading-endpoints)
- [Exercise 6 – Blueprints and Application Factory](#exercise-6--blueprints-and-application-factory)
- [Exercise 7 – Error Handling and Custom Responses](#exercise-7--error-handling-and-custom-responses)
- [Exercise 8 – Testing Flask Apps](#exercise-8--testing-flask-apps)
- [Exercise 9 – Background Work and Async Considerations](#exercise-9--background-work-and-async-considerations)
- [Exercise 10 – Project Structure](#exercise-10--project-structure)

In [1]:
import os, sys, numpy, asyncio, aiohttp, re, time, nest_asyncio
from typing import Any, List, Tuple, Dict, Set, Callable
from pandas import Series, DataFrame, Index, concat
from pandas import Timestamp, Timedelta, date_range
from dataclasses import dataclass
from aiohttp import ClientSession
from enum import Enum

from flask import Flask, Blueprint, request, jsonify
from flask.testing import FlaskClient
from pydantic import BaseModel, Field
from apids import APIDS

APIDS.load_config("./apids.json")

[08:28:06.532 | apids.load_config @ L57] Loaded config from ./apids.json


In [2]:
symbol_specs = list[Any]()
async with ClientSession() as session:
    for exchange, data in APIDS.CONFIG.items():
        if not (endpoint := "symbol_specs") in data: continue
        if (exchange == "Bybit"): etc = "linear"
        elif (exchange == "OKX"): etc = "SWAP"
        elif (exchange == "Deribit"): etc = "USDT"
        else: etc = None
        response = await APIDS.fetch(session, exchange, endpoint, etc)
        if "output" not in response: continue
        if len(response["output"]) == 0: continue
        symbol_specs.append(response["output"]) 

SPECS = concat(symbol_specs)
SPECS["point_size"] = numpy.pow(0.1, SPECS["digits"])

[08:28:07.292 | apids.fetch @ L222] Fetched Binance None
[08:28:08.626 | apids.fetch @ L222] Fetched Bybit linear
[08:28:09.152 | apids.fetch @ L222] Fetched OKX SWAP
[08:28:15.091 | apids.fetch @ L222] Fetched Kraken None
[08:28:15.526 | apids.fetch @ L222] Fetched Deribit USDT
[08:28:29.135 | apids.fetch @ L222] Fetched GateIO None


## Exercise 1 – Hello World Route

**Goal**: Create a minimal Flask application.

**Requirements**:

- Instantiate `app = Flask(__name__)` and add `GET /` returning a plain text greeting.
- Add `GET /ping` returning JSON `{"status": "ok"}`.
- Use `app.test_client()` to call both routes and assert status 200.

**Hint**: Set `app.config['TESTING'] = True` before using the test client.

In [4]:
app = Flask("Exercise_1")
app.config["TESTING"] = True

@app.route("/", methods = ["GET"])
def hello():
    args = request.get_json()
    name = args["name"]
    return {"message": f"Hello {name}! :)"}

with app.test_client() as client:
    resp = client.get("/", json = {"name": "Coco"})
    print(resp.json)
    assert resp.status_code == 200

{'message': 'Hello Coco! :)'}


## Exercise 2 – URL Path and Query Parameters

**Goal**: Accept parameters from the URL.

**Requirements**:

- Add `GET /symbols/<ticker>` that returns the ticker uppercased as JSON.
- Add a `precision: int = 2` query parameter that controls decimal rounding on a price.
- Test with a missing `ticker` (404) and a valid one (200).

**Hint**: Use `request.args.get('key', default)` for query parameters.

In [5]:
class API2:
    APP = Flask("Exercise_2")
    APP.config["TESTING"] = True

    @APP.route("/symbol/<path:symbol>", methods = ["GET"])
    def price(symbol: str):
        exchange, symbol = symbol.split("_")
        async def func():
            async with ClientSession() as session:
                response = await APIDS.fetch(session,
                    exchange, "current_price", symbol)
            df: DataFrame = response["output"]
            df.index = df.index.map(" ".join)
            response["output"] = df.to_dict(orient = "index")
            return response
        
        nest_asyncio.apply()
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(func())
                    
with API2.APP.test_client() as client:
    symbol = "Binance_BTC/USDT"
    resp = client.get(f"/symbol/{symbol}")
    assert resp.status_code == 200
    print(f"Call \"symbol({symbol})\":", resp.json["output"])

[08:28:36.070 | apids.fetch @ L222] Fetched Binance BTC/USDT


Call "symbol(Binance_BTC/USDT)": {'Binance BTC/USDT': {'ask': 71751.42, 'bid': 71751.41}}


## Exercise 3 – JSON Endpoints and the Request Object

**Goal**: Read and return JSON bodies.

**Requirements**:

- Add `POST /orders` that reads `request.get_json()` and returns the body with an added `id` field.
- Return 400 with a message if the body is missing or malformed.
- Test with `client.post('/orders', json={...})`.

**Hint**: `request.get_json(silent=True)` returns `None` instead of raising on bad JSON.

In [6]:
from aiohttp import ClientSession

class API3:
    APP = Flask("Exercise_3")
    APP.config["TESTING"] = True

    @APP.route("/symbols", methods = ["GET"])
    def prices():
        symbols = request.get_json()["symbols"]
        async def func():
            async with ClientSession() as session:
                tasks = list()
                for exc_sym in symbols:
                    exchange, symbol = str.split(exc_sym, "_")
                    tasks.append(asyncio.create_task(APIDS.fetch(
                      session, exchange, "current_price", symbol)))

                status = True
                output, errors = list(), list()
                for result in await asyncio.gather(*tasks):
                    if result["status"]:
                        df: DataFrame = result["output"]
                        df.index = df.index.map(" ".join)
                        output.append(df)
                    else: errors.extend(result["errors"])

            output = concat(output).to_dict(orient = "index")
            return {"status": status, "output": output, "errors": errors}

        nest_asyncio.apply()
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(func())
                    
with API3.APP.test_client() as client:
    symbols = ["Binance_XRP/USDT", "Bybit_XRP/USDT"]
    resp = client.get("/symbols", json = {"symbols": symbols})
    assert resp.status_code == 200
    print(f"Call \"symbols{(*symbols,)}\":", resp.json["output"])

[08:28:52.233 | apids.fetch @ L222] Fetched Binance XRP/USDT
[08:28:52.253 | apids.fetch @ L222] Fetched Bybit XRP/USDT


Call "symbols('Binance_XRP/USDT', 'Bybit_XRP/USDT')": {'Binance XRP/USDT': {'ask': 1.3811, 'bid': 1.381}, 'Bybit XRP/USDT': {'ask': 1.381, 'bid': 1.3809}}


## Exercise 4 – Data Validation with Pydantic

**Goal**: Use Pydantic to validate request data inside a Flask route.

**Requirements**:

- Define `OrderIn(BaseModel)` with `symbol`, `qty: int`, `price: float`.
- In the POST `/orders` route, call `OrderIn(**request.get_json())` inside a try/except `ValidationError`.
- Return 422 with the validation errors on failure.

**Hint**: Pydantic `ValidationError.errors()` returns a list of dicts with `loc`, `msg`, and `type`.

In [125]:
from datetime import datetime, timezone

class OrderForm(BaseModel):
    quantity: float = Field(...)
    exchange: str = Field(...) ; symbol: str = Field(...)
    price: float = Field(None) ; comment: str = Field(None)

class Order(OrderForm):
    ts: datetime = Field(...)
    side: str = Field(...)
    id: str = Field(...)    

    @classmethod
    def from_request(cls, request: OrderForm):
        try: 
            specs: Series = SPECS.loc[request.exchange]
            specs = specs.loc[request.symbol]
            min_qty = specs["min_trade_size"]
        except: min_qty = 0.001
        if abs(request.quantity) < min_qty:
            error = f"\"{abs(request.quantity):.2f} < {min_qty}\""
            raise ValueError(f"Quantity below min: " + error)
        quantity = round(abs(request.quantity) / min_qty) * min_qty
        side = "BUY" if (request.quantity > 0) else "SELL"
        ts = datetime.now(timezone.utc)
        tsi = int(ts.timestamp() * 1e6)
        id = numpy.base_repr(tsi, 36)
        comment = request.comment
        if not comment: comment = ""
        return cls(ts = ts, id = id, quantity = quantity, side = side,
                  exchange = request.exchange, symbol = request.symbol,
                  price = request.price, comment = comment)
class API4:
    APP = Flask("Exercise_4")
    APP.config["TESTING"] = True

    @APP.route("/order", methods = ["POST"])
    def order():
        payload: dict = request.get_json(
            force = True, silent = True)
        order = OrderForm(**payload)
        try: output = Order.from_request(order)
        except Exception as EXC: output = EXC
        success = isinstance(output, Order)
        status = 201 if success else 422
        output = output.model_dump() if success else repr(EXC)
        return {"success": success, "output": output}, status

with API4.APP.test_client() as client:
    payload = dict[str, Any](quantity = 10.0,
        exchange = "Binance", symbol = "BTCUSDT",
        price = 71000.0, comment = "test_0001")
    resp = client.post("/order", json = payload)
    assert resp.status_code == 201
    print("Order result:", resp.json)

Order result: {'output': {'comment': 'test_0001', 'exchange': 'Binance', 'id': 'HHETOZCQNI', 'price': 71000.0, 'quantity': 10.0, 'side': 'BUY', 'symbol': 'BTCUSDT', 'ts': 'Wed, 08 Apr 2026 09:17:47 GMT'}, 'success': True}


## Exercise 5 – In-Memory Trading Endpoints

**Goal**: Build a small stateful API backed by a Python dict.

**Requirements**:

- Maintain `ORDERS: dict[int, dict]` at module level.
- Implement `GET /orders`, `POST /orders`, `GET /orders/<id>`, `DELETE /orders/<id>`.
- Return 404 for unknown ids; return the list sorted by id.

**Hint**: `abort(404)` raises a `404 Not Found` response from anywhere in a route.

In [142]:
from itertools import product
exchanges = ["Binance", "Bybit", "OKX", "GateIO"]
symbols = "BTC ETH XRP AVAX LTC DOGE NEO BNB ADA TRX NANO DASH EOS IOTA XLM XMR LINK"
symbols = Index(sorted(symbols.split(" "))) + "/USDT"
symbols = sorted(map("_".join, product(exchanges, symbols)))

with API3.APP.test_client() as client:
    resp = client.get("/symbols", json = {"symbols": symbols})
    df = DataFrame.from_dict(resp.json["output"], orient = "index").sort_index()
    df.index = df.index.map(str.split).map(tuple).rename(["exchange", "symbol"])

orders = df.copy()["ask"].to_frame("price")
orders = orders.dropna().sample(20, replace = True)
orders["random"] = (numpy.random.random(orders.shape) * 2 - 1) / 100
orders["price"] = orders["price"] + orders.pop("random") * orders["price"]
orders["random"] = (numpy.random.random(orders.shape) * 2 - 1)
orders["quantity"] = (1000 + 9000 * orders.pop("random")) / orders["price"]
orders: DataFrame = orders.loc[orders["price"] > 0]
orders["quantity"] = orders["quantity"].round(3)
orders["price"] = orders["price"].round(6)
orders["comment"] = range(1, orders.shape[0] + 1)
orders["comment"] = orders["comment"].map("test_{:04.0f}".format)

[09:26:32.269 | apids.fetch @ L222] Fetched Binance LINK/USDT
[09:26:32.272 | apids.fetch @ L222] Fetched Binance NEO/USDT
[09:26:32.275 | apids.fetch @ L222] Fetched Binance EOS/USDT
[09:26:32.278 | apids.fetch @ L222] Fetched OKX LINK/USDT
[09:26:32.281 | apids.fetch @ L222] Fetched OKX XMR/USDT
[09:26:32.284 | apids.fetch @ L222] Fetched OKX BNB/USDT
[09:26:32.293 | apids.fetch @ L222] Fetched OKX XRP/USDT
[09:26:32.294 | apids.fetch @ L222] Fetched OKX NANO/USDT
[09:26:32.296 | apids.fetch @ L222] Fetched Binance LTC/USDT
[09:26:32.299 | apids.fetch @ L222] Fetched Binance BTC/USDT
[09:26:32.300 | apids.fetch @ L222] Fetched Binance XMR/USDT
[09:26:32.302 | apids.fetch @ L222] Fetched Binance TRX/USDT
[09:26:32.304 | apids.fetch @ L222] Fetched Binance NANO/USDT
[09:26:32.306 | apids.fetch @ L222] Fetched Binance BNB/USDT
[09:26:32.308 | apids.fetch @ L222] Fetched Binance DASH/USDT
[09:26:32.311 | apids.fetch @ L222] Fetched OKX LTC/USDT
[09:26:32.313 | apids.fetch @ L222] Fetched

In [143]:
class API5:
    APP = Flask("Exercise_5")
    APP.config["TESTING"] = True
    ORDERS = DataFrame(columns = Order.model_fields.keys())
    
    @APP.route("/order", methods = ["POST"])
    def order():
        payload: dict = request.get_json(
            force = True, silent = True)
        order = OrderForm(**payload)
        try: output = Order.from_request(order)
        except Exception as EXC: output = EXC
        success = isinstance(output, Order)
        status = 201 if success else 422
        if success: API5.ORDERS.loc[output.id] = output.__dict__
        output = output.model_dump() if success else repr(output)
        return {"success": success, "output": output}, status

    @APP.route("/orders", methods = ["GET"])
    def orders():
        since_str: str = request.args.get("since", None)
        until_str: str = request.args.get("until", None)
        symbols: list[str] = request.args.get("symbols", None)
        if not until_str: until = datetime.now(timezone.utc)
        else: until: datetime = datetime.strptime(until_str)
        if not since_str: since = datetime.fromtimestamp(0, timezone.utc)
        else: since: datetime = datetime.strptime(since_str)
        result: DataFrame = API5.ORDERS.copy()
        result = result.loc[result["ts"].between(since, until)]
        if symbols: result = result.loc[result["symbol"].isin(symbols)]
        return result.to_dict(orient = "index")

with API5.APP.test_client() as client:
    for (exchange, symbol), args in orders.iterrows():
        resp = client.post("/order", json = {**args,
            "exchange": exchange, "symbol": symbol})
        time.sleep(numpy.random.random() * 0.25)
    
    resp = client.get("/orders")
    result = DataFrame.from_dict(resp.json, orient = "index")
    result = result.set_index("id").sort_index()
    display(result)

,comment,exchange,price,quantity,side,symbol,ts
id,,,,,,,
HHETXT822D,test_0001,Bybit,0.094956,39536.714,SELL,DOGE/USDT,"Wed, 08 Apr 2026 09:26:41 GMT"
HHETXT8NW9,test_0002,OKX,0.163748,9704.982,SELL,XLM/USDT,"Wed, 08 Apr 2026 09:26:41 GMT"
HHETXTCJIJ,test_0003,Binance,54.779620,132.481,BUY,LTC/USDT,"Wed, 08 Apr 2026 09:26:41 GMT"
HHETXTECUA,test_0004,OKX,55.464803,48.632,SELL,LTC/USDT,"Wed, 08 Apr 2026 09:26:41 GMT"
HHETXTIDGG,test_0005,Bybit,55.239479,116.513,SELL,LTC/USDT,"Wed, 08 Apr 2026 09:26:41 GMT"
HHETXTJO44,test_0006,Binance,616.624122,13.089,BUY,BNB/USDT,"Wed, 08 Apr 2026 09:26:41 GMT"
HHETXTOG0F,test_0007,Bybit,71324.958022,0.064,SELL,BTC/USDT,"Wed, 08 Apr 2026 09:26:42 GMT"
HHETXTRTAG,test_0008,GateIO,9.286825,1018.821,BUY,LINK/USDT,"Wed, 08 Apr 2026 09:26:42 GMT"
HHETXTSL53,test_0009,OKX,0.060712,29953.687,SELL,IOTA/USDT,"Wed, 08 Apr 2026 09:26:42 GMT"


## Exercise 6 – Blueprints and Application Factory

**Goal**: Split routes across Blueprints and use the app factory pattern.

**Requirements**:

- Create a `market_bp = Blueprint('market', __name__, url_prefix='/market')` with one route.
- Create `orders_bp` with another route.
- Write `create_app(config=None)` that registers both blueprints and returns the app.

**Hint**: The factory pattern lets you create multiple app instances with different configs.

In [144]:
class API6:

    ORDERS = DataFrame(columns = Order.model_fields.keys())
    BP = Blueprint("orders", "Exercise_6", url_prefix = "/orders")
    
    @BP.route("/send", methods = ["POST"])
    def order():
        payload: dict = request.get_json(
            force = True, silent = True)
        order = OrderForm(**payload)
        try: output = Order.from_request(order)
        except Exception as EXC: output = EXC
        success = isinstance(output, Order)
        status = 201 if success else 422
        if success: API6.ORDERS.loc[output.id] = output.__dict__
        output = output.model_dump() if success else repr(output)
        return {"success": success, "output": output}, status

    @BP.route("/list", methods = ["GET"])
    def orders():
        since_str: str = request.args.get("since", None)
        until_str: str = request.args.get("until", None)
        symbols: list[str] = request.args.get("symbols", None)
        if not until_str: until = datetime.now(timezone.utc)
        else: until: datetime = datetime.strptime(until_str)
        if not since_str: since = datetime.fromtimestamp(0, timezone.utc)
        else: since: datetime = datetime.strptime(since_str)
        result: DataFrame = API6.ORDERS.copy()
        result = result.loc[result["ts"].between(since, until)]
        if symbols: result = result.loc[result["symbol"].isin(symbols)]
        return result.to_dict(orient = "index")

    APP = Flask("Exercise_6")
    APP.config["TESTING"] = True
    APP.register_blueprint(BP)

with API6.APP.test_client() as client:
    for (exchange, symbol), args in orders.iterrows():
        resp = client.post("/orders/send", json = {**args,
            "exchange": exchange, "symbol": symbol})
        time.sleep(numpy.random.random() * 0.25)
    
    resp = client.get("/orders/list")
    orders = DataFrame.from_dict(resp.json, orient = "index")
    orders = orders.set_index("id").sort_index()
    display(orders)

,comment,exchange,price,quantity,side,symbol,ts
id,,,,,,,
HHETY1C4CH,test_0001,Bybit,0.094956,39536.714,SELL,DOGE/USDT,"Wed, 08 Apr 2026 09:26:54 GMT"
HHETY1HD8X,test_0002,OKX,0.163748,9704.982,SELL,XLM/USDT,"Wed, 08 Apr 2026 09:26:55 GMT"
HHETY1IR8H,test_0003,Binance,54.779620,132.481,BUY,LTC/USDT,"Wed, 08 Apr 2026 09:26:55 GMT"
HHETY1LMT6,test_0004,OKX,55.464803,48.632,SELL,LTC/USDT,"Wed, 08 Apr 2026 09:26:55 GMT"
HHETY1MLXG,test_0005,Bybit,55.239479,116.513,SELL,LTC/USDT,"Wed, 08 Apr 2026 09:26:55 GMT"
HHETY1R0HC,test_0006,Binance,616.624122,13.089,BUY,BNB/USDT,"Wed, 08 Apr 2026 09:26:55 GMT"
HHETY1RDPW,test_0007,Bybit,71324.958022,0.064,SELL,BTC/USDT,"Wed, 08 Apr 2026 09:26:55 GMT"
HHETY1UOIH,test_0008,GateIO,9.286825,1018.821,BUY,LINK/USDT,"Wed, 08 Apr 2026 09:26:55 GMT"
HHETY1XRL3,test_0009,OKX,0.060712,29953.687,SELL,IOTA/USDT,"Wed, 08 Apr 2026 09:26:55 GMT"


## Exercise 7 – Error Handling and Custom Responses

**Goal**: Return consistent error shapes.

**Requirements**:

- Register `@app.errorhandler(404)` and `@app.errorhandler(500)` returning JSON `{"error": ...}`.
- Raise `abort(404)` inside a route and confirm the handler fires.
- Write a helper `json_error(code, message)` reused by all handlers.

**Hint**: `@app.errorhandler(Exception)` catches all unhandled exceptions.

## Exercise 8 – Testing Flask Apps

**Goal**: Write a complete test suite using the test client.

**Requirements**:

- Write `TestOrdersAPI(unittest.TestCase)` using `setUp` to create a fresh app and client.
- Test create, list, get, and delete operations.
- Assert HTTP status codes, JSON keys, and state changes.

**Hint**: Use the app factory from Exercise 6 with a `TESTING=True` config for isolation.

## Exercise 9 – Background Work and Async Considerations

**Goal**: Understand Flask's synchronous nature and workarounds.

**Requirements**:

- Add a `POST /reports/generate` endpoint that simulates slow work with `time.sleep(0.1)`.
- Run it with `threading.Thread(target=generate_report, daemon=True).start()` to return immediately.
- Explain why a real deployment would use Celery or RQ instead.

**Hint**: Flask itself is synchronous by default; `async def` routes require an ASGI server.

## Exercise 10 – Project Structure

**Goal**: Organise a production-ready Flask project.

**Requirements**:

- Print the recommended layout: `app/__init__.py` (factory), `app/routes/`, `app/models.py`, `config.py`.
- Show how `config.py` defines `DevelopmentConfig` and `ProductionConfig` classes.
- List three Flask-specific best practices.

**Hint**: Use `flask run` with `FLASK_ENV=development` for local dev; never expose the dev server publicly.